# Step 22 — Second backbone: ResNet34d at 224px

**Settings: `GPU T4 x2`, `Internet On`.**

**Inputs:** competition data, `rsna-knee-13a-cache224lat-sagcor`, `rsna-knee-13b-cache224lat-axial`, `rsna-knee-03-extract-all`

## What happened with ConvNeXt, and what changed

ConvNeXt-nano collapsed to gold 0.55 — barely above chance. Pretrained weights loaded correctly, so the cause was the **learning rate**: ConvNeXt uses LayerNorm with no squeeze-excitation and is normally fine-tuned at 1e-5 to 1e-4. The OneCycle peak of 3e-4 destroyed the pretrained features in the first epoch. EfficientNet's BatchNorm tolerates that peak; ConvNeXt doesn't.

**ResNet34d is BatchNorm-based**, so it works with the recipe that already produced 0.878 — no retuning, no gamble.

## Why ResNet34d for diversity

| | EfficientNet-B0 | ResNet34d |
|---|---|---|
| params | 5.3M | 21.3M |
| convolutions | depthwise-separable | standard 3x3 |
| stem | 3x3 | **7x7** |
| squeeze-excitation | yes | no |
| scaling | compound | plain depth |

Different feature extractor, different failure modes. That's what makes an ensemble worth building — a model scoring slightly lower but erring *differently* contributes more than a near-twin scoring slightly higher.

## The sanity gate

Cell 5 trains **one epoch on fold 0 and stops**. If val AUC after one epoch is below 0.65, the backbone isn't learning and the notebook halts — 20 minutes lost instead of 5 hours. ConvNeXt would have been caught here.

Only after that passes do you run the full folds.

### 1 · Configure

In [ ]:
BACKBONE = "resnet34d"
TARGET_FOLDS = [0, 1]          # -> then [2, 3] -> then [4]
EPOCHS, BS, ACCUM, LR = 8, 3, 2, 3e-4
SANITY_ONLY = True             # <-- True = one epoch then stop. Set False for the real run.
print(f"{BACKBONE} | folds {TARGET_FOLDS} | sanity_only={SANITY_ONLY}")

### 2 · Setup

In [ ]:
import os, gc, glob, time, math, random, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, cv2, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score

def seed_all(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
arrA = np.load(glob.glob("/kaggle/input/notebooks/*/*/cache224lat_a.npy")[0], mmap_mode="r")
arrB = np.load(glob.glob("/kaggle/input/notebooks/*/*/cache224lat_b.npy")[0], mmap_mode="r")
PSEUDO = glob.glob("/kaggle/input/notebooks/*/*/pseudo_labels.csv")[0]
GOLD   = glob.glob("/kaggle/input/notebooks/*/*/gold58.csv")[0]

N, _, NSL, PX, _ = arrA.shape
NSER = arrA.shape[1] + arrB.shape[1]
LABELS = pd.read_csv(f"{ROOT}/sample_submission.csv").columns[1:].tolist()
KEYS   = ["acl","mcl","medial_meniscus","lateral_meniscus","medial_oa","lateral_oa",
          "pf_oa","effusion","synovitis","bakers","contusion","fracture"]

train  = pd.read_csv(f"{ROOT}/train.csv")
pseudo = pd.read_csv(PSEUDO)
gold   = pd.read_csv(GOLD)
assert (pseudo["StudyInstanceUID"].values == train["StudyInstanceUID"].values).all()

Y = pseudo[KEYS].values.astype(np.float32) / 100.0
nan_mask = np.isnan(Y).any(1)
Y = np.nan_to_num(Y, nan=0.2).astype(np.float32)
print("A", arrA.shape, "B", arrB.shape, "| GPU:", torch.cuda.get_device_name(0))

### 3 · Splits and loss weights — unchanged from every previous run

In [ ]:
gold_ids = set(gold["StudyInstanceUID"])
is_gold  = train["StudyInstanceUID"].isin(gold_ids).values

t0 = time.time()
present = np.ones((N, NSER), np.float32)
for i in range(N):
    for k in range(2):
        present[i, k] = float(arrA[i, k, NSL // 2].max() > 0)
    present[i, 2] = float(arrB[i, 0, NSL // 2].max() > 0)
print(f"presence in {time.time()-t0:.0f}s | mean series/study {present.sum(1).mean():.2f}")

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
except ImportError:
    os.system("pip install -q iterative-stratification")
    from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

usable = np.where((~is_gold) & (present.sum(1) > 0) & (~nan_mask))[0]
Ybin   = (Y[usable] > 0.5).astype(int)
mskf   = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds  = np.zeros(len(usable), int)
for f, (_, v) in enumerate(mskf.split(usable, Ybin)):
    folds[v] = f
gd_idx = np.where(is_gold & (present.sum(1) > 0))[0]
print("fold sizes:", [int((folds == f).sum()) for f in range(5)])

gl_ids  = gold["StudyInstanceUID"].tolist()
Pg_ext  = np.nan_to_num(pseudo.set_index("StudyInstanceUID").loc[gl_ids, KEYS].values.astype(np.float32), nan=20.0)
Yg_true = gold.set_index("StudyInstanceUID").loc[gl_ids, LABELS].values.astype(int)
ext_auc = np.array([roc_auc_score(Yg_true[:, c], Pg_ext[:, c]) for c in range(12)])
w = ext_auc - 0.5; w = w / w.mean()
loss_w = torch.tensor(w, dtype=torch.float32).cuda()

### 4 · Dataset and model

In [ ]:
class KneeDS(Dataset):
    def __init__(self, idx, train_mode):
        self.idx, self.train_mode = idx, train_mode
    def __len__(self): return len(self.idx)
    def __getitem__(self, n):
        i = self.idx[n]
        v = np.concatenate([np.asarray(arrA[i]), np.asarray(arrB[i])], 0).astype(np.float32) / 255.0
        if self.train_mode:
            ang   = np.random.uniform(-12, 12)
            scale = np.random.uniform(0.9, 1.1)
            tx    = np.random.uniform(-0.06, 0.06) * PX
            ty    = np.random.uniform(-0.06, 0.06) * PX
            M = cv2.getRotationMatrix2D((PX/2, PX/2), ang, scale)
            M[0, 2] += tx; M[1, 2] += ty
            gain = np.random.uniform(0.85, 1.15)
            bias = np.random.uniform(-0.08, 0.08)
            for s in range(NSER):
                if v[s].max() == 0: continue
                for j in range(NSL):
                    v[s, j] = cv2.warpAffine(v[s, j], M, (PX, PX), borderValue=0)
                v[s] = np.clip(v[s] * gain + bias, 0, 1)
        return torch.from_numpy(v), torch.from_numpy(present[i]), torch.from_numpy(Y[i])

def loader(idx, train_mode, bs=BS):
    return DataLoader(KneeDS(idx, train_mode), batch_size=bs, shuffle=train_mode,
                      num_workers=2, pin_memory=True, drop_last=train_mode,
                      persistent_workers=True)

import timm

class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))
    def forward(self, x, mask=None):
        a = self.w(x).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(mask < 0.5, -1e4)
        return (x * a.softmax(-1).unsqueeze(-1)).sum(1)

class Net(nn.Module):
    def __init__(self, backbone=BACKBONE, nlab=12, hid=256, pretrained=True):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, in_chans=3)
        d = self.enc.num_features
        self.gru        = nn.GRU(d, hid, batch_first=True, bidirectional=True)
        self.slice_pool = AttnPool(hid * 2)
        self.ser_pool   = AttnPool(hid * 2)
        self.head       = nn.Sequential(nn.Dropout(0.3), nn.Linear(hid * 2, nlab))
    def forward(self, v, mask):
        B, S, T, H, W = v.shape
        lo = torch.cat([v[:, :, :1], v[:, :, :-1]], 2)
        hi = torch.cat([v[:, :, 1:], v[:, :, -1:]], 2)
        x  = torch.stack([lo, v, hi], 3).reshape(B * S * T, 3, H, W)
        f = self.enc(x).reshape(B * S, T, -1)
        f, _ = self.gru(f)
        f = self.slice_pool(f).reshape(B, S, -1)
        return self.head(self.ser_pool(f, mask))

def macro_auc(P, T, thresh=0.5):
    P = np.nan_to_num(P, nan=0.5); out = []
    for c in range(T.shape[1]):
        y = (T[:, c] > thresh).astype(int)
        out.append(np.nan if y.min() == y.max() else roc_auc_score(y, P[:, c]))
    return np.nanmean(out), out

_m = Net().cuda()
print("encoder features:", _m.enc.num_features,
      "| params:", round(sum(p.numel() for p in _m.parameters())/1e6, 2), "M")
del _m; torch.cuda.empty_cache()

### 5 · Train, with a sanity gate

After epoch 0, val AUC must exceed **0.65**. EfficientNet-B0 reached ~0.72 after one epoch; ConvNeXt never got past 0.53. Anything below the threshold means the backbone isn't learning under this recipe, and the run aborts.

With `SANITY_ONLY = True` it stops after that first epoch regardless — 20 minutes to find out whether ResNet works.

In [ ]:
SANITY_MIN = 0.65

def train_fold(FOLD, sanity_only=False):
    ck = f"/kaggle/working/rn34_fold{FOLD}.pt"
    if os.path.exists(ck) and not sanity_only:
        print(f"fold {FOLD}: exists, skipping"); return None

    seed_all(42 + FOLD)
    tr_idx, va_idx = usable[folds != FOLD], usable[folds == FOLD]
    tl, vl, gl = loader(tr_idx, True), loader(va_idx, False), loader(gd_idx, False)

    model  = Net().cuda()
    opt    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    n_ep   = 1 if sanity_only else EPOCHS
    sched  = torch.optim.lr_scheduler.OneCycleLR(opt, LR,
                total_steps=EPOCHS * (len(tl) // ACCUM + 1), pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda")
    lossf  = nn.BCEWithLogitsLoss(weight=loss_w)

    @torch.no_grad()
    def ev(dl):
        model.eval(); P, T = [], []
        for v, mk, y in dl:
            with torch.amp.autocast("cuda"):
                p = model(v.cuda(non_blocking=True), mk.cuda())
            P.append(p.float().sigmoid().cpu().numpy()); T.append(y.numpy())
        return np.concatenate(P), np.concatenate(T)

    print(f"\n{'='*58}\nFOLD {FOLD}  train {len(tr_idx):,} val {len(va_idx):,}"
          f"{'  [SANITY]' if sanity_only else ''}\n{'='*58}", flush=True)
    best, best_pv = 0, None
    for ep in range(n_ep):
        model.train(); tot = 0; t0 = time.time()
        opt.zero_grad(set_to_none=True)
        for bi, (v, mk, y) in enumerate(tl):
            v, mk, y = v.cuda(non_blocking=True), mk.cuda(), y.cuda()
            with torch.amp.autocast("cuda"):
                loss = lossf(model(v, mk), y) / ACCUM
            scaler.scale(loss).backward()
            if (bi + 1) % ACCUM == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step()
                opt.zero_grad(set_to_none=True)
            if not math.isfinite(loss.item()):
                print("non-finite loss, aborting"); return None
            tot += loss.item() * ACCUM
            if bi % 400 == 0:
                el = time.time() - t0
                print(f"  ep{ep} {bi}/{len(tl)} loss {tot/(bi+1):.4f} eta {(len(tl)-bi)*el/(bi+1)/60:.0f}m", flush=True)

        Pv, Tv = ev(vl); av, _ = macro_auc(Pv, Tv)
        Pg, Tg = ev(gl); ag, _ = macro_auc(Pg, Tg)
        print(f"EPOCH {ep}  loss {tot/len(tl):.4f}  val {av:.4f}  gold {ag:.4f}  [{(time.time()-t0)/60:.0f}m]", flush=True)

        if ep == 0:
            if av < SANITY_MIN:
                print(f"\n*** SANITY FAILED: val {av:.4f} < {SANITY_MIN} ***")
                print("*** backbone is not learning under this recipe - do NOT run full folds ***")
                return av
            print(f"   sanity OK ({av:.4f} >= {SANITY_MIN})")
            if sanity_only:
                del model, tl, vl, gl; gc.collect(); torch.cuda.empty_cache()
                return av

        if av > best:
            best, best_pv = av, Pv
            torch.save({"model": {k: t.cpu() for k, t in model.state_dict().items()},
                        "fold": FOLD, "val_auc": av, "gold_auc": ag, "epoch": ep,
                        "px": PX, "nsl": NSL, "backbone": BACKBONE}, ck)

    if best_pv is not None:
        o = pd.DataFrame(best_pv, columns=LABELS)
        o.insert(0, "StudyInstanceUID", train["StudyInstanceUID"].values[va_idx])
        o.to_csv(f"/kaggle/working/oofrn34_fold{FOLD}.csv", index=False)
        print(f"fold {FOLD} best val {best:.4f}")
    del model, tl, vl, gl; gc.collect(); torch.cuda.empty_cache()
    return best


T0 = time.time()
if SANITY_ONLY:
    av = train_fold(0, sanity_only=True)
    print(f"\nsanity result: val after 1 epoch = {av:.4f}" if av else "\nsanity aborted")
    print("B0 reference after 1 epoch: ~0.72 | ConvNeXt managed only 0.53")
    print("\nIf this passed, set SANITY_ONLY = False and commit the real run.")
else:
    for f in TARGET_FOLDS:
        train_fold(f)
print(f"\nelapsed {(time.time()-T0)/60:.0f} min")

### 6 · Compare against EfficientNet-B0

Only runs once real checkpoints exist. Reference: B0 singles 0.815, 5-model ensemble 0.835, leaderboard 0.878.

In [ ]:
paths = sorted(glob.glob("/kaggle/working/rn34_fold*.pt"))
print(f"ResNet34d checkpoints: {len(paths)}")

if paths:
    gl = loader(gd_idx, False)
    allP, singles = [], []
    for p in paths:
        c = torch.load(p, map_location="cpu", weights_only=False)
        m = Net(pretrained=False).cuda(); m.load_state_dict(c["model"]); m.eval()
        P, T = [], []
        with torch.no_grad():
            for v, mk, y in gl:
                with torch.amp.autocast("cuda"):
                    o = m(v.cuda(non_blocking=True), mk.cuda())
                P.append(o.float().sigmoid().cpu().numpy()); T.append(y.numpy())
        P, T = np.concatenate(P), np.concatenate(T)
        a, _ = macro_auc(P, T); singles.append(a); allP.append(P)
        print(f"  fold {c['fold']}: gold {a:.4f}")
        del m; gc.collect(); torch.cuda.empty_cache()

    ae, per = macro_auc(np.mean(allP, 0), T)
    print(f"\nResNet34d singles : {np.mean(singles):.4f}   (B0 0.815)")
    print(f"ResNet34d ensemble: {ae:.4f}   (B0 0.835)")
    print("\nper-label:")
    for l, a in zip(LABELS, per):
        print(f"  {l:<18} {a:.3f}")